### Data Reading

### create catalog first and create volume inside catalog and add .csv file there and then use dbutils.fs.ls() to find the file location

In [0]:
%sql
CREATE VOLUME databricks_tutorial.default.my_volume;

In [0]:
dbutils.fs.ls('/Volumes/databricks_tutorial/default/my_volume/')

### read csv file

In [0]:
from pyspark.sql.functions import *
from pyspark.sql.types import *
df=spark.read.format('csv').option('inferSchema',True).option('header',True).load("/Volumes/databricks_tutorial/default/my_volume/BigMart Sales (1).csv")
df.display()

### reading json

In [0]:
df2=spark.read.format('json').option('header',True).option('inferSchema',True).option('multiline',False).load('/Volumes/databricks_tutorial/default/my_volume/drivers.json')
df2.display()

In [0]:
df.printSchema()

### ddl schema

In [0]:
# DDL Schema
my_ddl_Schema='''
    Item_Identifier string,
    Item_Weight string,
    Item_Fat_Content string,
    Item_Visibility double,
    Item_Type string,
    Item_MRP double,
    Outlet_Identifier string,
    Outlet_Establishment_Year int,
    Outlet_Size string,
    Outlet_Location_Type string,
    Outlet_Type string,
    Item_Outlet_Sales double
'''

In [0]:
df1=spark.read.format('csv').schema(my_ddl_Schema).option('header',True).load('/Volumes/databricks_tutorial/default/my_volume/BigMart Sales (1).csv')

In [0]:
df1.display()


### structtype() schema

In [0]:
from pyspark.sql.types import * 
from pyspark.sql.functions import * 

In [0]:
my_strct_schema=StructType([
    StructField('Item_Identifier',StringType(),True),
    StructField('Item_weight',StringType(),True),
    StructField('Item_Fat_Content',StringType(),True),
    StructField('Item_Visibility',StringType(),True),
    StructField('Item_Type',StringType(),True),
    StructField('Item_MRP',StringType(),True),
    StructField('Outlet_Identifier',StringType(),True),
    StructField('Outlet_Establishment_Year',StringType(),True),
    StructField('Outlet_Size',StringType(),True),
    StructField('Outlet_Location_Type',StringType(),True),
    StructField('Outlet_Type',StringType(),True),
    StructField('Item_Outlet_Sales',StringType(),True)
])

In [0]:
df2=spark.read.format('csv').schema(my_strct_schema).option('header',True).load('/Volumes/databricks_tutorial/default/my_volume/BigMart Sales (1).csv')
df2.display()

In [0]:
df.display()

### select specific columns

In [0]:
df.select('Item_Type','Item_MRP').display()

In [0]:
# other way to extract col
df.select(col('Item_Type'),col('Item_MRP')).display()

### alias function

In [0]:
df.select(col('Item_Identifier').alias('Item_ID')).display()

### filter and where

![image_1780015860150.png](./image_1780015860150.png "image_1780015860150.png")

In [0]:
df.display()


### scenerio -1

In [0]:
df.filter(col('Item_Fat_Content')=='Regular').display()

### scenerio-2

In [0]:
df.filter((col('Item_Type')=='Soft Drinks') & (col('Item_Weight')<10)).display()

### scenerio-3    

#### isnull() and isin()

In [0]:
df.filter((col('Outlet_size').isNull()) & (col('Outlet_Location_Type').isin('Tier 1','Tier 2'))).display()


### withcolumnRename -> to rename columns

In [0]:
df.withColumnRenamed('Item_Weight','Item_wt').display()


### with column -> modify existing column and creating new columns

In [0]:
from pyspark.sql.functions import lit
df.withColumn('flag',lit("new")).display()

In [0]:
from pyspark.sql.functions import *
df.withColumn('multiply',col('Item_Weight')*col('Item_MRP')).display()

### modifying existing column

#### regexp_replace()

In [0]:
df.withColumn('Item_Fat_Content',regexp_replace('Item_Fat_Content','Regular','Reg'))\
    .withColumn('Item_Fat_Content',regexp_replace('Item_Fat_Content','Low Fat','Lf')).display()

#### Type Casting -> Uses very often in case of apply joins aggregation

#### cast()

In [0]:
df=df.withColumn('Item_Weight',col('Item_Weight').cast(StringType()))

In [0]:
df.printSchema()

### sort/orderby

#### scenerio-1 .sort()/.asc().desc()

In [0]:
# sort column item_weight in desc
df.sort(col('Item_Weight').desc()).display()

#### scenerio -2

In [0]:
df.sort(col('Item_Visibility').asc()).display()

#### sorting based on multiple columns

In [0]:
# both item_weight and item_visibilty n desc
df.sort(['Item_Weight','Item_Visibility'],ascending=[0,0]).display()

In [0]:
df.sort(['Item_Weight','Item_Visibility'],ascending=[0,1]).display()

### limit

In [0]:
df.limit(10).display()

### drop

In [0]:
df.drop('Item_Visibility').display()

### scenario-2

In [0]:
df.drop('Item_Visibility','Item_Type').display()

### drop_duplicates

In [0]:
df.dropDuplicates().display()

In [0]:
df.dropDuplicates(subset=['Item_Type']).display()

In [0]:
# to get distinct values
df.distinct().display()

### union and union byname

In [0]:
data1 = [('1','kad'),
        ('2','sid')]
schema1 = 'id STRING, name STRING' 

df1 = spark.createDataFrame(data1,schema1)

data2 = [('3','rahul'),
        ('4','jas')]
schema2 = 'id STRING, name STRING' 

df2 = spark.createDataFrame(data2,schema2)

In [0]:
df1.display()

In [0]:
df2.display()

In [0]:
df1.union(df2).display()

In [0]:
data1 = [('kad','1',),
        ('sid','2')]
schema1 = 'name STRING,id STRING' 

df1 = spark.createDataFrame(data1,schema1)

In [0]:
df1.display()

In [0]:
df1.union(df2).display()

In [0]:
df1.unionByName(df2).display()


### string functions
#### init_cap, upper, lower

In [0]:
df.select(initcap('Item_type')).display()

In [0]:
df.select(upper('Item_type')).display()

In [0]:
df.select(lower('Item_type')).display()

In [0]:
df.select(upper('Item_type').alias('upper_item_type')).display()

### Date Functions

#### Current_date // date_add() // date_sub()

In [0]:
df=df.withColumn('curr_date',current_date())

In [0]:
df.display()

### date_add()

In [0]:
## add one week to current date and call it as like future_date
df=df.withColumn('week_after',date_add('curr_date',7))

In [0]:
df.display()

### date_sub

In [0]:
df.withColumn('week_before',date_sub('curr_date',7)).display()

In [0]:
df=df.withColumn('week_before',date_add('curr_date',-7))
df.display()

### datediff

In [0]:
df=df.withColumn('datediff',datediff('week_after','curr_date'))

In [0]:
df.display()

### date_format

In [0]:
df=df.withColumn('week_before',date_format('week_before','dd-MM-yyyy'))
df.display()

### handling nulls

#### dropping nulls / filling nulls

#### how = any -> in each column if any null it will drop that record/all -> drop all column nulls whole record ie whole row

In [0]:
df.dropna(how='all').display()

In [0]:
df.dropna(how='any').display()

In [0]:
# using subset is better
df.dropna(subset=['Outlet_Size']).display()

In [0]:
df.display()

### fill null values

In [0]:
df.fillna('NotAvailable').display()

In [0]:
df.fillna('Not Available',subset=['Outlet_Size']).display()

### split and indexing

In [0]:
df.withColumn('Outlet_Type',split('Outlet_Type',' ')).display()

### index

In [0]:
df.withColumn('Outlet_Type',split('Outlet_Type',' ')[1]).display()

### Explode

![image_1780115733695.png](./image_1780115733695.png "image_1780115733695.png")

In [0]:
df_exp=df.withColumn('Outlet_Type',split('Outlet_Type',' '))
df_exp.display()

In [0]:
df_exp.withColumn('Outlet_Type',explode('Outlet_Type')).display()

### array_contains

In [0]:
# to know type1 avaialbale in outlet_type list -> that colmn in array
df_exp.withColumn('Type1_flag',array_contains('Outlet_Type','Type1')).display()


### groupBy -> most important

In [0]:
df.groupby('Item_Type').agg(sum('Item_MRP')).display()

In [0]:
df.groupby('Item_Type').agg(avg('Item_MRP')).display()

### groupby multiple columns

In [0]:
df.groupby('Item_Type','Outlet_Size').agg(sum('Item_MRP').alias('Total_price')).display()

In [0]:
## also include avg mrp along with total
df.groupby('Item_Type','Outlet_Size').agg(sum('Item_MRP').alias('Total_MRP'),avg('Item_MRP').alias('Avg_MRP')).display()

### Collect_list -> similar to group_concat() in MySQL

In [0]:
data=[
    ('user1','book1'),
    ('user1','book2'),
    ('user2','book2'),
    ('user2','book4'),
    ('user3','book1'),
]

schema='user string,book string'

df_book=spark.createDataFrame(data,schema)

df_book.display()

In [0]:
df_book.groupBy('user').agg(collect_list('book')).display()

### PIVOT

In [0]:
df.groupBy('Item_Type').pivot('Outlet_Size').agg(avg('Item_MRP')).display()

### when-otherwise like if else// case when

In [0]:
df=df.withColumn('Veg_flag',when(col('Item_Type')=='Meat','Non-Veg').otherwise('Veg'))
df.display()

In [0]:
df.withColumn('veg_exp_flag',when(((col('Veg_flag')=='Veg') & (col('Item_MRP')<100)),'Veg_Inexpensive')\
                            .when((col('Veg_flag')=='Veg')& (col('Item_MRP')>=100),'Veg_Expensive')\
                            .otherwise('Non_Veg')).display()

## Joins

In [0]:
dataj1 = [('1', 'gaur', 'd01'),
          ('2', 'kit', 'd02'),
          ('3', 'sam', 'd03'),
          ('4', 'tim', 'd03'),
          ('5', 'aman', 'd05')]

schemaj1 = 'emp_id STRING, emp_name STRING, dept_id STRING'

df1 = spark.createDataFrame(dataj1, schemaj1)

dataj2 = [('d01', 'HR'),
          ('d02', 'Marketing'),
          ('d03', 'Accounts'),
          ('d04', 'IT'),
          ('d05', 'Finance')]

schemaj2 = 'dept_id STRING, department STRING'

df2 = spark.createDataFrame(dataj2, schemaj2)


In [0]:
df1.display()

In [0]:
df2.display()